# 07 — Research Synthesis, Reproducibility & Final Deliverables

**Project:** Stellar Kinematics Mapping of a Galaxy Using MaNGA IFU Data  
**Target:** Plate-IFU `8138-12704`  
**DAPTYPE:** `HYB10-MILESHC-MASTARSSP`

## Purpose
This milestone turns the analysis outputs into a reproducible research package. It does not invent or pre-fill scientific results. It checks which upstream outputs exist, records file provenance and software versions, collects figures and numerical summaries, and builds a structured final-report draft with clearly marked fields that require your measured results and scientific interpretation.

## Required upstream work
Run notebooks 03–06 first. Some outputs are optional; missing files will be recorded rather than silently fabricated.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import json
import hashlib
import platform
import sys
import importlib.metadata as metadata
from datetime import datetime, timezone

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"D:\\KS\\manga-stellar-kinematics"),
]
PROJECT_ROOT = next(
    (p.resolve() for p in candidate_roots if (p / "data" / "raw").is_dir()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT manually to your project directory.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
REPORTS_DIR = OUTPUT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())

Project root: D:\KS\manga-stellar-kinematics
Python: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
Platform: Windows-10-10.0.26300-SP0


## 2. Check expected source data and analysis outputs

In [2]:
expected_files = {
    "MAPS": RAW_DIR / "manga-8138-12704-MAPS-HYB10-MILESHC-MASTARSSP.fits.gz",
    "DRP_LOGCUBE": RAW_DIR / "manga-8138-12704-LOGCUBE.fits.gz",
    "DAP_LOGCUBE": RAW_DIR / "manga-8138-12704-LOGCUBE-HYB10-MILESHC-MASTARSSP.fits.gz",
    "processed_kinematics": PROCESSED_DIR / "stellar_kinematics_8138-12704.npz",
    "processed_geometry": PROCESSED_DIR / "manga_geometry_8138-12704.npz",
    "milestone5_summary": OUTPUT_DIR / "stellar_kinematics_quantitative_summary.txt",
    "milestone5_radial_profiles": OUTPUT_DIR / "stellar_kinematics_radial_profiles.csv",
    "milestone6_robustness": OUTPUT_DIR / "robustness_uncertainty_report.txt",
    "milestone6_report_draft": OUTPUT_DIR / "research_report_draft_8138-12704.md",
}

status = {}
for name, path in expected_files.items():
    status[name] = {"exists": path.exists(), "path": str(path)}
    print(f"{'FOUND' if path.exists() else 'MISSING':7s} {name:28s} {path}")

missing_required = [
    name for name in ["MAPS", "processed_kinematics",
                      "milestone5_summary", "milestone6_robustness"]
    if not status[name]["exists"]
]
if missing_required:
    print("\nWARNING — required outputs are missing:", missing_required)
    print("Complete the corresponding earlier notebooks before finalizing the report.")

FOUND   MAPS                         D:\KS\manga-stellar-kinematics\data\raw\manga-8138-12704-MAPS-HYB10-MILESHC-MASTARSSP.fits.gz
FOUND   DRP_LOGCUBE                  D:\KS\manga-stellar-kinematics\data\raw\manga-8138-12704-LOGCUBE.fits.gz
FOUND   DAP_LOGCUBE                  D:\KS\manga-stellar-kinematics\data\raw\manga-8138-12704-LOGCUBE-HYB10-MILESHC-MASTARSSP.fits.gz
FOUND   processed_kinematics         D:\KS\manga-stellar-kinematics\data\processed\stellar_kinematics_8138-12704.npz
FOUND   processed_geometry           D:\KS\manga-stellar-kinematics\data\processed\manga_geometry_8138-12704.npz
FOUND   milestone5_summary           D:\KS\manga-stellar-kinematics\outputs\stellar_kinematics_quantitative_summary.txt
FOUND   milestone5_radial_profiles   D:\KS\manga-stellar-kinematics\outputs\stellar_kinematics_radial_profiles.csv
FOUND   milestone6_robustness        D:\KS\manga-stellar-kinematics\outputs\robustness_uncertainty_report.txt
FOUND   milestone6_report_draft      D:\KS\manga-s

## 3. Inspect upstream summary files without assuming their values

In [3]:
def read_text_if_exists(path):
    return path.read_text(encoding="utf-8") if path.exists() else None

milestone5_text = read_text_if_exists(expected_files["milestone5_summary"])
milestone6_text = read_text_if_exists(expected_files["milestone6_robustness"])
milestone6_draft = read_text_if_exists(expected_files["milestone6_report_draft"])

print("===== MILESTONE 5 SUMMARY =====")
print(milestone5_text if milestone5_text else "Not generated yet.")

print("\n===== MILESTONE 6 ROBUSTNESS REPORT =====")
print(milestone6_text if milestone6_text else "Not generated yet.")

print("\n===== MILESTONE 6 REPORT DRAFT =====")
print(milestone6_draft if milestone6_draft else "Not generated yet.")

===== MILESTONE 5 SUMMARY =====
MaNGA Stellar Kinematics Quantitative Summary
Plate-IFU: 8138-12704
DAPTYPE: HYB10-MILESHC-MASTARSSP
Unique stellar bins: 2194
Valid velocity bins: 2194
Valid corrected-dispersion bins: 2194
Joint valid bins: 2194
Residual systemic velocity estimate [km/s]: 33.6441
Robust velocity scatter [km/s]: 190.3056
First-order kinematic PA [deg, N through E, modulo 180]: 167.3003
Velocity gradient amplitude [km/s/arcsec]: 20.2767
Projected rotation amplitude proxy [km/s]: 206.8342
lambda_R within 1 Re: 0.64621
lambda_R within 2 Re: 0.71560
Caution: these are first-pass projected estimates, not deprojected dynamical quantities.

===== MILESTONE 6 ROBUSTNESS REPORT =====
MaNGA Robustness and Uncertainty Report
Plate-IFU: 8138-12704
DAPTYPE: HYB10-MILESHC-MASTARSSP
Generated UTC: 2026-11-04T18:25:08.222583+00:00
Bootstrap iterations requested: 2000
Random seed: 813812704

STANDARD QUALITY SAMPLE
n_bins: 2194
v_sys_residual_kms: 33.64405632019043
kinematic_pa_deg_mod1

## 4. Inventory figures generated by previous milestones

In [4]:
figure_files = sorted(FIGURES_DIR.glob("*")) if FIGURES_DIR.exists() else []
figure_inventory = []
for path in figure_files:
    if path.is_file():
        figure_inventory.append({
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "modified_utc": datetime.fromtimestamp(
                path.stat().st_mtime, tz=timezone.utc
            ).isoformat(),
        })

print("Figure files found:", len(figure_inventory))
for item in figure_inventory:
    print(f"- {item['filename']} ({item['size_bytes']} bytes)")

Figure files found: 16
- bootstrap_kinematic_pa_deg_mod180.png (62533 bytes)
- bootstrap_lambda_R_1Re.png (56965 bytes)
- bootstrap_lambda_R_2Re.png (52041 bytes)
- bootstrap_vrot_projected_proxy_kms.png (65348 bytes)
- elliptical_geometry_maps.png (130105 bytes)
- lambda_R_cumulative_profile.png (66783 bytes)
- manga_flux_snr_context.png (129360 bytes)
- milestone9_dispersion_radial_profile.png (45475 bytes)
- milestone9_velocity_radial_profile.png (30450 bytes)
- spaxel_sky_offsets.png (110930 bytes)
- stellar_kinematics_radial_profiles.png (158013 bytes)
- stellar_sigma_corrected_map.png (88099 bytes)
- stellar_sigma_corrected_map_wcs.png (86161 bytes)
- stellar_velocity_bin_residuals.png (423824 bytes)
- stellar_velocity_map_standard.png (76846 bytes)
- stellar_velocity_map_wcs.png (71685 bytes)


## 5. Record input-file metadata and SHA-256 checksums

In [5]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

input_manifest = []
for key in ["MAPS", "DRP_LOGCUBE", "DAP_LOGCUBE"]:
    path = expected_files[key]
    if path.exists():
        input_manifest.append({
            "role": key,
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

print(json.dumps(input_manifest, indent=2))

[
  {
    "role": "MAPS",
    "filename": "manga-8138-12704-MAPS-HYB10-MILESHC-MASTARSSP.fits.gz",
    "size_bytes": 11000233,
    "sha256": "e2ddf65e99df64be21aa6d0970cbf18aa103ca62d548a0682cacbf85c43ba130"
  },
  {
    "role": "DRP_LOGCUBE",
    "filename": "manga-8138-12704-LOGCUBE.fits.gz",
    "size_bytes": 201542147,
    "sha256": "a2cf8bf1244d3f0594bd77c2b5e46aa0666f21256ac1fffe7d763b3ba41b9a69"
  },
  {
    "role": "DAP_LOGCUBE",
    "filename": "manga-8138-12704-LOGCUBE-HYB10-MILESHC-MASTARSSP.fits.gz",
    "size_bytes": 224743906,
    "sha256": "f3580555416fa0eb4bc8ba39eb25646029642e886b827fc166c2415670962664"
  }
]


## 6. Capture Python package versions

In [6]:
packages_to_record = [
    "numpy", "scipy", "matplotlib", "astropy",
    "pandas", "jupyter", "ipykernel"
]
package_versions = {}
for package in packages_to_record:
    try:
        package_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        package_versions[package] = "NOT INSTALLED"

environment_record = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "packages": package_versions,
}
print(json.dumps(environment_record, indent=2))

{
  "python_version": "3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]",
  "platform": "Windows-10-10.0.26300-SP0",
  "packages": {
    "numpy": "2.4.6",
    "scipy": "1.17.1",
    "matplotlib": "3.11.2",
    "astropy": "8.0.2",
    "pandas": "3.1.0",
    "jupyter": "1.1.1",
    "ipykernel": "7.4.0"
  }
}


## 7. Create a machine-readable project manifest

In [7]:
manifest = {
    "project": "Stellar Kinematics Mapping of a Galaxy Using MaNGA IFU Data",
    "target": {
        "plate_ifu": "8138-12704",
        "daptype": "HYB10-MILESHC-MASTARSSP",
        "survey_release": "SDSS-IV MaNGA DR17 (confirm exact release metadata from source headers)",
    },
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "input_files": input_manifest,
    "expected_outputs": status,
    "figures": figure_inventory,
    "environment": environment_record,
    "analysis_notes": [
        "Stellar kinematics in HYB10 are associated with stellar-continuum Voronoi bins.",
        "Stellar velocity is line-of-sight and relative to the DAP input guess redshift.",
        "Corrected stellar dispersion must account for the instrumental correction.",
        "Bootstrap intervals in this project quantify sampling variation only.",
        "Large-scale stellar kinematics alone do not establish an SMBH binary."
    ],
}

manifest_path = REPORTS_DIR / "project_manifest_8138-12704.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", manifest_path)

Saved: D:\KS\manga-stellar-kinematics\outputs\reports\project_manifest_8138-12704.json


## 8. Build the final report skeleton

In [8]:
report_sections = [
    ("Abstract", "[Write a concise summary only after validating the numerical results.]"),
    ("1. Introduction", "[Introduce MaNGA IFU spectroscopy, stellar kinematics, and the scientific motivation.]"),
    ("2. Target and Data", "[Record target selection, plate-IFU, survey release, data products, and DAPTYPE.]"),
    ("3. Data Reduction and Quality Control", "[Describe MAPS/LOGCUBE products, stellar binning, mask flags, and inverse-variance selection.]"),
    ("4. Methods", "[Describe velocity extraction, instrumental dispersion correction, coordinate system, plane fit, radial profiles, lambda_R, and bootstrap.]"),
    ("5. Results", "[Insert measured values and refer to the actual generated figures. Do not invent values.]"),
    ("6. Robustness and Uncertainty", "[Compare standard/strict cuts and report bootstrap intervals plus systematic limitations.]"),
    ("7. Discussion", "[Interpret the observed velocity and dispersion structure only as supported by the data.]"),
    ("8. Relevance to the Candidate SMBH-Binary Host-Galaxy Question", "[Explain what large-scale stellar kinematics constrain and what they cannot establish.]"),
    ("9. Limitations", "[Discuss PSF/beam smearing, inclination, adopted geometry, template mismatch, binning, coverage, and systematics.]"),
    ("10. Conclusion", "[Summarize the defensible findings and propose justified follow-up observations.]"),
    ("References", "[Add official SDSS/MaNGA data-model and methods references actually consulted.]"),
]

report_md = [
    "# Stellar Kinematics Mapping of MaNGA Target 8138-12704",
    "",
    "**DAPTYPE:** `HYB10-MILESHC-MASTARSSP`",
    "",
    "**Status:** Research-report template — complete after validating outputs.",
    "",
]
for heading, body in report_sections:
    report_md.extend([f"## {heading}", "", body, ""])

report_path = REPORTS_DIR / "final_research_report_template_8138-12704.md"
report_path.write_text("\n".join(report_md), encoding="utf-8")
print("Saved:", report_path)

Saved: D:\KS\manga-stellar-kinematics\outputs\reports\final_research_report_template_8138-12704.md


## 9. Create a figure-caption worksheet

In [9]:
caption_rows = [
    ("stellar_velocity_map_wcs.png",
     "Stellar line-of-sight velocity field in celestial coordinates. State the adopted quality mask, velocity reference, and color scale."),
    ("stellar_sigma_corrected_map_wcs.png",
     "Instrument-corrected stellar velocity dispersion. State the correction convention and quality mask."),
    ("elliptical_geometry_maps.png",
     "DAP-provided elliptical radius and azimuth maps. State the coordinate conventions and adopted geometry."),
    ("stellar_kinematics_radial_profiles.png",
     "Radial profiles of median residual velocity and corrected stellar velocity dispersion, binned in elliptical effective-radius units."),
    ("lambda_R_cumulative_profile.png",
     "Cumulative projected lambda_R as a function of elliptical aperture radius. State the flux weighting and valid-bin selection."),
]

caption_path = REPORTS_DIR / "figure_caption_worksheet_8138-12704.md"
caption_text = ["# Figure Caption Worksheet", ""]
for filename, caption in caption_rows:
    exists = (FIGURES_DIR / filename).exists()
    caption_text += [
        f"## {filename}",
        f"**Figure file present:** {'Yes' if exists else 'No'}",
        caption,
        "[After inspecting the actual figure, add target-specific observations here.]",
        "",
    ]
caption_path.write_text("\n".join(caption_text), encoding="utf-8")
print("Saved:", caption_path)

Saved: D:\KS\manga-stellar-kinematics\outputs\reports\figure_caption_worksheet_8138-12704.md


## 10. Final quality-control gate

In [10]:
qc_checks = {
    "MAPS input present": expected_files["MAPS"].exists(),
    "processed kinematics present": expected_files["processed_kinematics"].exists(),
    "Milestone 5 numerical summary present": expected_files["milestone5_summary"].exists(),
    "Milestone 6 robustness report present": expected_files["milestone6_robustness"].exists(),
    "WCS velocity figure present": (FIGURES_DIR / "stellar_velocity_map_wcs.png").exists(),
    "corrected sigma figure present": (FIGURES_DIR / "stellar_sigma_corrected_map_wcs.png").exists(),
    "radial profile figure present": (FIGURES_DIR / "stellar_kinematics_radial_profiles.png").exists(),
    "lambda_R figure present": (FIGURES_DIR / "lambda_R_cumulative_profile.png").exists(),
}

for name, passed in qc_checks.items():
    print(f"{'PASS' if passed else 'PENDING':8s} {name}")

print("\nThis is a file-presence check, not scientific validation.")
print("Scientific validation requires inspecting the actual figures, numerical values, masks and assumptions.")

PASS     MAPS input present
PASS     processed kinematics present
PASS     Milestone 5 numerical summary present
PASS     Milestone 6 robustness report present
PASS     WCS velocity figure present
PASS     corrected sigma figure present
PASS     radial profile figure present
PASS     lambda_R figure present

This is a file-presence check, not scientific validation.
Scientific validation requires inspecting the actual figures, numerical values, masks and assumptions.


## 11. Milestone 7 checklist

- [ ] All upstream notebooks 03–06 have been executed successfully.
- [ ] Numerical summaries have been inspected and verified.
- [ ] Figure inventory checked against the actual figures.
- [ ] Input data checksums and software environment recorded.
- [ ] Machine-readable project manifest generated.
- [ ] Final report skeleton completed using actual results.
- [ ] Figure captions revised after inspecting each plot.
- [ ] Scientific claims reviewed for evidence and limitations.

## Deliverables created by this notebook

```text
outputs/
└── reports/
    ├── project_manifest_8138-12704.json
    ├── final_research_report_template_8138-12704.md
    └── figure_caption_worksheet_8138-12704.md
```

## What happens after this milestone?

The computational workflow is organized. The next stage is not simply another calculation: it is **result validation and scientific writing**. We should review your actual maps, summary metrics, bootstrap intervals, and target metadata before making a conclusion about the galaxy or its relevance to a candidate SMBH-binary host.